In [ ]:
# import the dependecies
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch import nn



In [ ]:
# Run device agnostic code
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Get the data
train_data = datasets.MNIST(root='/data',
                            train=True,
                            transform=ToTensor(),
                            target_transform=None,
                            download=True
                            )

test_data = datasets.MNIST(root='/data',
                           train=False,
                           transform=ToTensor(),
                           target_transform=None,
                           download=True
                           )

print(len(train_data), len(test_data))


In [ ]:
# See more about the data (See shape, Create batches and more)
train_data[0]

In [ ]:
class_names = train_data.classes
class_names

In [ ]:
img , label = train_data[0]
print(img.shape)
print(img.squeeze(0).shape)

In [ ]:
plt.title(class_names[label])
plt.imshow(img.squeeze(0), cmap='gray')
plt.axis(False)

In [ ]:
BATCH_SIZE = 32


train_dataloader = DataLoader(dataset=train_data,
                              batch_size=BATCH_SIZE,
                              shuffle=True
                              )

test_dataloader = DataLoader(dataset=test_data,
                             batch_size=BATCH_SIZE,
                             shuffle=False
                             )

print(len(train_dataloader), len(test_dataloader))

In [ ]:
# Build The CNN Model

class CNN_Model(nn.Module):
  def __init__(self, input_shape: int, hidden_layers: int, output_shape: int):
    super().__init__()
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=input_shape, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2)
    )

    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2)
    )

    self.classifier_layer = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_layers * 7 * 7, out_features=output_shape)
    )

  def forward(self, x):
    return self.classifier_layer(self.conv_block_2(self.conv_block_1(x)))


In [ ]:
len(class_names)

In [ ]:
model = CNN_Model(input_shape=1, hidden_layers=10, output_shape=len(class_names)).to(device)
print(model)

In [ ]:
# Training and Testing the Model

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.1)

def accuracy_fn(y_true, y_pred):
  correct = torch.eq(y_true, y_pred).sum().item()
  acc = correct / len(y_pred) * 100

  return acc

In [ ]:
from tqdm.auto import tqdm
torch.manual_seed(17) #ooooh Kelvin Debryune


epochs = 3
train_loss = 0
train_acc = 0
test_acc = 0
test_loss = 0


for epoch in tqdm(range(epochs)):
  print(f'Epoch: {epoch}')

  for batch, (X, Y) in enumerate(train_dataloader):
    X, Y = X.to(device), Y.to(device)

    # Training mode
    model.train()

    # forward pass
    y_predictions = model(X)

    # Loss
    loss = loss_fn(y_predictions, Y)

    # Calculate predictions for accuracy (correct way for multi-class with CrossEntropyLoss)
    y_preds = y_predictions.argmax(dim=1)

    train_acc += accuracy_fn(y_true=Y, y_pred=y_preds)
    train_loss += loss

    # Zero Gradient
    optimizer.zero_grad()

    # Back Propagation
    loss.backward()

    # Optimizer Step
    optimizer.step()

    if batch % 400 == 0:
      print(f'Looked at {batch * len(X)} / {len(train_dataloader.dataset)} smaples')

  train_loss /= len(train_dataloader)
  train_acc /= len(train_dataloader)

  # Test mode
  with torch.inference_mode():
    model.eval()

    for X, Y in test_dataloader:
      X, Y = X.to(device), Y.to(device)

      y_predictions = model(X)

      loss = loss_fn(y_predictions, Y)
      # Calculate predictions for accuracy (correct way for multi-class with CrossEntropyLoss)
      y_preds = y_predictions.argmax(dim=1)

      test_acc += accuracy_fn(y_true=Y, y_pred=y_preds)
      test_loss += loss

    test_acc /= len(test_dataloader)
    test_loss /= len(test_dataloader)

  print(f'Train Loss: {train_loss:.4f} || Train Accuracy: {train_acc:.2f}% || Test Loss: {test_loss:.4f} || Test Accuracy: {test_acc:.2f}%')

In [ ]:
# Save the Model
from pathlib import Path

Model_File_path = Path('model')
Model_File_path.mkdir(parents=True, exist_ok=True)

Model_Name = 'MNIST_model.pth'
Model_Save_Path = Model_File_path / Model_Name

torch.save(obj=model.state_dict(), f=Model_Save_Path)

In [ ]:
# Downloading Model to Downloads from Google Colab

# from google.colab import files
# files.download('./model/MNIST_model.pth')